# Dataset and Basic Code

In [ ]:
!gdown --id 14d266Azey8AkTQyzj3FUL_AJ6YGoEgRd

!unzip /content/cropped.zip

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import os
import matplotlib.pyplot as plt
import cv2
import random
# import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models, optimizers
import seaborn as sns
from collections import Counter
from tensorflow import keras
from tensorflow.keras import layers
from keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from tensorflow.keras import callbacks
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPool2D, BatchNormalization, DepthwiseConv2D
from tensorflow.keras.models import Model
# from tensorflow.keras.optimizers import RMSprop
# from keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, CSVLogger
# import winsound
from time import time
from datetime import datetime

In [ ]:
# img = cv2.imread("/content/cropped/001Walid/01.jpg", 0)
# # img = cv2.imread("/content/SdataSet/001Walid/01.jpg", 0)
# plt.imshow(img, cmap= 'gray')

In [ ]:
DATADIR = r'/content/cropped'
CATEGORIES = []
for folder in sorted(os.listdir(DATADIR)):
    if folder == ".DS_Store":
        continue
    CATEGORIES.append(folder)
print(len(CATEGORIES))

#Separate Testing and Training Data
IMG_SIZE = 150
training_data_masked = []
training_data_unmasked = []
testing_data = []
# flatten_training_data = []
# target_array = []

def create_training_data():
  for catagory in CATEGORIES:
    if catagory[0] == '.':
        continue

    path = os.path.join(DATADIR,catagory)

    class_num = CATEGORIES.index(catagory)

    for img in os.listdir(path):

        if img[0] == '.' :
            continue
#         print(img)
#         print(catagory, img)
        img_array = cv2.imread(os.path.join(path, img))
        new_array = cv2.resize(img_array, (IMG_SIZE, IMG_SIZE))


        # Adding Filter
#         new_array = cv2.Canny(new_array,100,150)

         #  The Image becomes 2D/Blackn White after filtering, but vgg19 requires a 3-channel image. So converting the 2d image to 3d
#         new_array = cv2.cvtColor(new_array, cv2.COLOR_GRAY2RGB)


        image_number = int(img[:2])



        # if image_number == 6:  #front blue masked
        #     testing_data.append([new_array, class_num])

        # elif image_number == 7:   # left blue masked
        #     testing_data.append([new_array, class_num])

        # elif image_number == 11:    # front black masked
        #     testing_data.append([new_array, class_num])

        # elif image_number == 13:    # right black masked
        #     testing_data.append([new_array, class_num])

        if image_number == 6:  #front blue masked
            testing_data.append([new_array, class_num])

        elif image_number == 11:   # left blue masked
            testing_data.append([new_array, class_num])

        elif image_number == 15:    # front black masked
            testing_data.append([new_array, class_num])

        elif image_number == 16:    # right black masked
            testing_data.append([new_array, class_num])

        else:
            if image_number in [1,2,3,4,5]:
                training_data_unmasked.append([new_array, class_num])
            else:
                training_data_masked.append([new_array, class_num])


t0 = time()
create_training_data()
print(f'Taken Time {time()-t0}')

264
Taken Time 73.94304275512695


In [ ]:
print(len(CATEGORIES))

264


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score


def get_results(model):

    results = {
#     "train_loss":0,
    "test_loss":0,
#     "train_acc":0,
    "test_acc":0,
    "recall":0,
    "precision":0,
    "f1":0,

    }

    y_pred = model.predict(X_test)
    y_pred_bool = []

    for y in y_pred:
      y_pred_bool.append(np.argmax(y))

#     y_pred_bool = np.array(y_pred_bool)
#     print(type(Y_test), type(y_pred_bool))
#     print(Y_test, y_pred_bool)

    results["test_loss"], results["test_acc"]= model.evaluate(X_test, Y_test, verbose = 0)

    results["precision"] = precision_score(Y_test, y_pred_bool, average = 'weighted')
    results["recall"] = recall_score (Y_test, y_pred_bool, average = 'weighted')
    results["f1"] = f1_score(Y_test, y_pred_bool, average = 'weighted')

    return results

In [ ]:
!mkdir Model
!mkdir Logs!mkdir Graphs
!mkdir Results

In [ ]:
# def get_name(model, t, acc = False):

#     model_save_path = r"/content/Model"
#     log_save_path = r"/content/Logs"
#     graph_save_path = r"/content/Graphs"
#     results_save_path = r"/content/Results"


#     if t=="graph":
#         if acc:
#             uni_name = os.path.join(graph_save_path, f"{model} Open Face ACC {get_date_time()}.jpg")
#         else:
#             uni_name = os.path.join(graph_save_path, f"{model} Open Face Loss {get_date_time()}.jpg")

#     elif t=="model":
#         uni_name = os.path.join(model_save_path, f"{model} Open Face {get_date_time()}.h5")
#     elif t=='log':
#         uni_name = os.path.join(log_save_path, f"{model} Open Face {get_date_time()}.log")
#     elif t== 'results':
#         uni_name = os.path.join(results_save_path, f"{model} Open Face {get_date_time()}.csv")
#     else:
#         raise ValueError(f'You entered invalid value')


#     return uni_name

# "Oracle-Shadow Latent Alignment" (OSLA)

In [ ]:
import tensorflow as tf
import numpy as np
import random

def build_osla_dataset(unmasked_list, masked_list, test_list):
    # 1. Map images to Identity to create matching pairs
    u_dict = {}
    for img, label in unmasked_list:
        if label not in u_dict: u_dict[label] = []
        u_dict[label].append(img)

    m_dict = {}
    for img, label in masked_list:
        if label not in m_dict: m_dict[label] = []
        m_dict[label].append(img)

    x_u_paired, x_m_paired, y_train_paired = [], [], []

    # 2. Innovative Pairing: Every masked image gets matched with a random unmasked 'Anchor' of the same person
    for label in m_dict.keys():
        if label in u_dict:
            for m_img in m_dict[label]:
                x_m_paired.append(m_img)
                x_u_paired.append(random.choice(u_dict[label]))
                y_train_paired.append(label)

    # 3. Normalize (0-1) and Convert to Tensors
    x_u_paired = np.array(x_u_paired) / 255.0
    x_m_paired = np.array(x_m_paired) / 255.0
    y_train_paired = np.array(y_train_paired)

    x_val = np.array([item[0] for item in test_list]) / 255.0
    y_val = np.array([item[1] for item in test_list])

    # 4. Create the final Dataset objects
    train_dataset = tf.data.Dataset.from_tensor_slices(((x_u_paired, x_m_paired), y_train_paired)).shuffle(1000).batch(32)
    validation_dataset = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(32)

    return train_dataset, validation_dataset, len(y_train_paired)

# Execute the builder
train_ds, val_ds, total_pairs = build_osla_dataset(training_data_unmasked, training_data_masked, testing_data)
print(f"OSLA Dataset Ready. Total Training Pairs: {total_pairs}")

OSLA Dataset Ready. Total Training Pairs: 2921


In [ ]:
from tensorflow.keras import layers, models, optimizers

base_vgg = tf.keras.applications.VGG16(weights='imagenet', include_top=False, input_shape=(150, 150, 3))
# We unfreeze the last 4 layers of VGG for high-precision fine-tuning
base_vgg.trainable = True
for layer in base_vgg.layers[:-4]:
    layer.trainable = False

# --- 2. Identity Extraction Network ---
inputs = layers.Input(shape=(150, 150, 3))
x = base_vgg(inputs)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.BatchNormalization()(x) # Essential for stability

# The Identity Embedding (Innovation)
# We calculate consistency HERE, not on the softmax
embedding = layers.Dense(512, activation=None, name="latent_embedding")(x)
embedding = layers.Lambda(lambda t: tf.math.l2_normalize(t, axis=1))(embedding)

# Final Prediction
outputs = layers.Dense(len(CATEGORIES), activation='softmax')(embedding)

# Create two outputs: one for classification, one for the embedding
osla_model = models.Model(inputs, [outputs, embedding])

# --- 3. Loss & Optimizer ---
# Increased learning rate slightly for faster convergence from 50%
optimizer = optimizers.Adam(learning_rate=1e-4)
loss_ce = tf.keras.losses.SparseCategoricalCrossentropy()
loss_mse = tf.keras.losses.MeanSquaredError()

@tf.function
def train_step(img_u, img_m, labels):
    with tf.GradientTape() as tape:
        # Get both Prediction and Embedding
        pred_u, emb_u = osla_model(img_u, training=True)
        pred_m, emb_m = osla_model(img_m, training=True)

        # 1. Classification Loss (Standard Cross-Entropy)
        # Weighting unmasked slightly more as it's the 'ground truth' identity
        c_loss = (loss_ce(labels, pred_u) * 0.6) + (loss_ce(labels, pred_m) * 0.4)

        # 2. INNOVATIVE LATENT CONSISTENCY (The Core Contribution)
        # We force the internal embeddings to be identical.
        # This is much more effective than prediction-level MSE.
        consistency_loss = loss_mse(emb_u, emb_m)

        # Total Loss: Balance classification and alignment
        total_loss = c_loss + (1.0 * consistency_loss)

    grads = tape.gradient(total_loss, osla_model.trainable_variables)
    optimizer.apply_gradients(zip(grads, osla_model.trainable_variables))
    return total_loss

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
import time

train_loss_history = []
train_acc_history  = []
val_loss_history   = []
val_acc_history    = []

print("Training OSLA with Latent Consistency...\n")

for epoch in range(30):
    start = time.time()

    # -------- Training --------
    train_losses = []
    train_acc = tf.keras.metrics.SparseCategoricalAccuracy()

    for (batch_u, batch_m), y in train_ds:
        loss = train_step(batch_u, batch_m, y)
        train_losses.append(loss)

        preds, _ = osla_model(batch_u, training=False)
        train_acc.update_state(y, preds)

    tr_loss = np.mean(train_losses)
    tr_acc  = train_acc.result().numpy()

    train_loss_history.append(tr_loss)
    train_acc_history.append(tr_acc)

    # -------- Validation --------
    val_losses = []
    val_acc = tf.keras.metrics.SparseCategoricalAccuracy()

    for x_val, y_val in val_ds:
        preds, _ = osla_model(x_val, training=False)

        v_loss = tf.keras.losses.sparse_categorical_crossentropy(y_val, preds)
        val_losses.append(tf.reduce_mean(v_loss).numpy())

        val_acc.update_state(y_val, preds)

    v_loss = np.mean(val_losses)
    v_acc  = val_acc.result().numpy()

    val_loss_history.append(v_loss)
    val_acc_history.append(v_acc)


    print(f"Epoch {epoch+1} | Training_Loss: {v_loss:.4f} | Training_Accuracy: {v_acc:.4f}  | Val_Loss: {v_loss:.4f} | Val_Accuracy: {v_acc:.4f} | Time: {time.time()-start:.1f}s")


In [ ]:
from sklearn.metrics import (accuracy_score, classification_report,
                             cohen_kappa_score, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             mean_squared_error, mean_absolute_error,
                             roc_curve)
from sklearn.preprocessing import label_binarize
from scipy.optimize import brentq
from scipy.interpolate import interp1d

# Final Evaluation
all_true, all_pred, all_probs = [], [], []

for v_img, v_label in val_ds:
    raw_preds, _ = osla_model.predict(v_img, verbose=0)
    preds = np.argmax(raw_preds, axis=1)

    all_probs.extend(raw_preds)
    all_pred.extend(preds)
    all_true.extend(v_label.numpy())

all_true  = np.array(all_true)
all_pred  = np.array(all_pred)
all_probs = np.array(all_probs)

# ---- Accuracy ----
accuracy         = accuracy_score(all_true, all_pred)
overall_accuracy = accuracy_score(all_true, all_pred)

# ---- Average Accuracy (per-class) ----
cm = confusion_matrix(all_true, all_pred)
per_class_acc    = cm.diagonal() / cm.sum(axis=1)
average_accuracy = np.mean(per_class_acc)

# ---- Kappa ----
kappa = cohen_kappa_score(all_true, all_pred)

# ---- Rank-1 Accuracy ----
rank1_accuracy = np.mean(all_pred == all_true)

# ---- EER ----
classes      = np.unique(all_true)
all_true_bin = label_binarize(all_true, classes=classes)
eers = []
for i in range(len(classes)):
    fpr, tpr, _ = roc_curve(all_true_bin[:, i], all_probs[:, i])
    frr = 1 - tpr
    eer = brentq(lambda x: interp1d(fpr, fpr - frr)(x), fpr[0], fpr[-1])
    eers.append(eer)
eer = np.mean(eers)

# ---- Precision / Recall / F1 ----
precision = precision_score(all_true, all_pred, average='weighted')
recall    = recall_score(all_true, all_pred, average='weighted')
f1        = f1_score(all_true, all_pred, average='weighted')

# ---- Loss Metrics ----
# mse       = mean_squared_error(all_true, all_pred)
test_loss = osla_model.evaluate(val_ds, verbose=0)[0]  # index 0 = loss

# ---- Print Results ----
print(f"  Accuracy:          {accuracy         * 100:.2f}%")
print(f"  Overall Accuracy:  {overall_accuracy * 100:.2f}%")
print(f"  Average Accuracy:  {average_accuracy * 100:.2f}%")
print(f"  Kappa:             {kappa:.4f}")
print(f"  Rank-1 Accuracy:   {rank1_accuracy   * 100:.2f}%")
print(f"  EER:               {eer              * 100:.2f}%")
print(f"  Precision:         {precision        * 100:.2f}%")
print(f"  Recall:            {recall           * 100:.2f}%")
print(f"  F1 Score:          {f1               * 100:.2f}%")
print(f"  MSE:               {mse:.4f}")
print(f"  Test Loss:         {test_loss:.4f}")

# ---- Classification Report ----
print("\nClassification Report:")
print(classification_report(all_true, all_pred, target_names=CATEGORIES))

In [ ]:
plt.figure(figsize=(10,8))
plt.plot(train_loss_history, color='teal', label="Training loss", marker='o')
plt.plot(val_loss_history, color='red', label="Validation loss", marker='o')

legend = plt.legend(loc='best', shadow=True, fontsize=12)

plt.xlabel('Epochs', fontsize=12, color='black')
plt.ylabel('Loss', fontsize=12, color='black')

plt.title("Training vs Validation Loss (OSLA)", fontsize=12, color='black')
plt.grid()

plt.savefig("OSLA_loss_curve.png", dpi=400)
plt.show()


In [ ]:
plt.figure(figsize=(10,8))
plt.plot(train_acc_history, color='teal', label="Training Accuracy", marker='o')
plt.plot(val_acc_history, color='red', label="Validation Accuracy", marker='o')

legend = plt.legend(loc='best', shadow=True, fontsize=12)

plt.xlabel('Epochs', fontsize=12, color='black')
plt.ylabel('Accuracy', fontsize=12, color='black')

plt.title("Training vs Validation Accuracy (OSLA)", fontsize=12, color='black')
plt.grid()

plt.savefig("OSLA_accuracy_curve.png", dpi=400)
plt.show()
